In [ ]:
import pandas as pd
import requests
import os
from cns.data_utils import data_path

# COSMIC

In [2]:
gene_table = pd.read_csv(f'{data_path}/Census_allFri Nov 17 10_27_09 2023.tsv', sep='\t')
gene_table.head()

,Gene Symbol,Name,Entrez GeneId,Genome Location,Tier,Hallmark,Chr Band,Somatic,Germline,Tumour Types(Somatic),Tumour Types(Germline),Cancer Syndrome,Tissue Type,Molecular Genetics,Role in Cancer,Mutation Types,Translocation Partner,Other Germline Mut,Other Syndrome,Synonyms
0,A1CF,APOBEC1 complementation factor,29974.0,10:50799421-50885675,2,NaN,11.23,yes,NaN,melanoma,NaN,NaN,E,NaN,oncogene,Mis,NaN,NaN,NaN,"ACF,ACF64,ACF65,APOBEC1CF,ASP,CCDS73133.1,ENSG..."
1,ABI1,abl-interactor 1,10006.0,10:26746593-26860935,1,Yes,12.10,yes,NaN,AML,NaN,NaN,L,Dom,"TSG, fusion",T,KMT2A,NaN,NaN,"ABI-1,CCDS7150.1,E3B1,ENSG00000136754.17,NM_00..."
2,ABL1,v-abl Abelson murine leukemia viral oncogene h...,25.0,9:130713946-130885683,1,Yes,34.12,yes,NaN,"CML, ALL, T-ALL",NaN,NaN,L,Dom,"oncogene, fusion","T, Mis","BCR, ETV6, NUP214",NaN,NaN,"ABL,CCDS35165.1,ENSG00000097007.17,JTK7,NM_007..."
3,ABL2,"c-abl oncogene 2, non-receptor tyrosine kinase",27.0,1:179099327-179229601,1,NaN,25.20,yes,NaN,AML,NaN,NaN,L,Dom,"oncogene, fusion",T,ETV6,NaN,NaN,"ABLL,ARG,CCDS30947.1,ENSG00000143322.19,NM_007..."
4,ACKR3,atypical chemokine receptor 3,57007.0,2:236569641-236582358,1,Yes,37.30,yes,NaN,lipoma,NaN,NaN,M,Dom,"oncogene, fusion",T,HMGA2,NaN,NaN,"CCDS2516.1,CMKOR1,CXCR7,ENSG00000144476.5,GPR1..."


In [3]:
def get_gene_position(gene_name):
    server = "https://grch37.rest.ensembl.org"
    ext = f"/lookup/symbol/homo_sapiens/{gene_name}?content-type=application/json"
    r = requests.get(server+ext, headers={"Content-Type": "application/json"})
    
    if not r.ok:
        return None, None, None

    decoded = r.json()
    return decoded['seq_region_name'], decoded['start'], decoded['end']

# Apply the function to each gene in gene_table
gene_table['chrom'], gene_table['start'], gene_table['end'] = zip(*gene_table['Gene Symbol'].map(get_gene_position))

In [4]:
genes = gene_table[['chrom', 'start', 'end', 'Gene Symbol']]
genes.head()

,chrom,start,end,Gene Symbol
0,10,52559169.0,52645435.0,A1CF
1,10,27035522.0,27150016.0,ABI1
2,9,133589333.0,133763062.0,ABL1
3,1,179068462.0,179198819.0,ABL2
4,2,237476430.0,237491001.0,ACKR3


In [5]:
# find rows with nan
nan_rows = genes[genes["start"].isna()]
len(nan_rows)

16

In [9]:
genes = genes[~genes["start"].isna()]
genes["start"] = genes["start"].astype(int) - 1
genes["end"] = genes["end"].astype(int)
genes.sort_values(by=["chrom", "start"], inplace=True)
genes["chrom"] = "chr" + genes["chrom"]
genes.rename(columns={"Gene Symbol": "name", "start": "chromStart", "end": "chromEnd"}, inplace=True)

In [13]:
genes.to_csv(f'{data_path}/COSMIC_consensus_genes.bed', sep="\t", index=False, header=False)

## PCAGW

The sample sheet is downloaded from https://dcc.icgc.org/releases/PCAWG/donors_and_biospecimens


The CN files are obtained by donwloading both `consensus.20170119.somatic.cns.icgc.public.tar.gz` and ` consensus.20170119.somatic.cna.tcga.public.tar.gz` from https://dcc.icgc.org/releases/PCAWG/consensus_cnv . 

In [ ]:
# !MUST HAVE THE INDIVIDUAL SAMPLES AND SAMPLE SHEET IN THE PCAWG FOLDER
pcawg_ids = pd.read_csv(f'{data_path}/PCAWG/pcawg_sample_sheet.tsv', sep='\t')

In [ ]:
directory = 'PCAWG'

pcawg_samples = []

for filename in os.listdir(directory):
    if filename.endswith(".txt"):
        df = pd.read_csv(os.path.join(directory, filename), sep='\t')   
        sample_id = filename[0:filename.find('.')]     
        df.insert(0, 'aliquot_id', sample_id)        
        pcawg_samples.append(df)
len(pcawg_samples)

In [ ]:
pcawg_dfs = pd.concat(pcawg_samples)
pcawg_dfs.head()

In [ ]:
mapping = pcawg_ids[["aliquot_id", "icgc_specimen_id"]]
map_dict = mapping.set_index('aliquot_id')['icgc_specimen_id'].to_dict()
pcawg_dfs['aliquot_id'] = pcawg_dfs['aliquot_id'].map(map_dict)
pcawg_dfs.head()

In [ ]:
pcawg_dfs = pcawg_dfs.rename(columns={'aliquot_id': 'sample_id', 'chromosome': 'chrom'})
pcawg_dfs['start'] = pcawg_dfs['start'].astype(int)
pcawg_dfs['end'] = pcawg_dfs['end'].astype(int)
pcawg_dfs['chrom'] = 'chr' + pcawg_dfs['chrom'].astype(str)
# drop star
pcawg_dfs = pcawg_dfs.drop(columns=['star', "total_cn"])

In [ ]:
# count rows in final_df that have NaN values
pcawg_dfs[pcawg_dfs.isna().any(axis=1)].shape[0]

In [ ]:
pcawg_dfs.to_csv(f'{data_path}/pcawg_cns_source.tsv', sep='\t', index=False)

## TGCA hg19

The CN files are in the archive: https://github.com/VanLoo-lab/ascat/raw/master/ReleasedData/TCGA_SNP6_hg19/segments.tar.gz .

In [ ]:
directory = 'tcga_hg19'

tcga_hg19_samples = []

for filename in os.listdir(f'{data_path}/{directory}'):
    if filename.endswith(".txt"):
        df = pd.read_csv(os.path.join(directory, filename), sep='\t')      
        tcga_hg19_samples.append(df)
len(tcga_hg19_samples)

In [ ]:
tcga_hg19_dfs = pd.concat(tcga_hg19_samples)
tcga_hg19_dfs.head()

In [ ]:
tcga_hg19_dfs.columns = ["sample_id", "chrom", "start", "end", "major_cn", "minor_cn"]
tcga_hg19_dfs['chrom'] = 'chr' + tcga_hg19_dfs['chrom'].astype(str)
tcga_hg19_dfs.head()

In [ ]:
tcga_hg19_dfs.to_csv(f'{data_path}/tcga_hg19_cns_source.tsv', sep='\t', index=False)

## TGCA hg38

The CN files are in the archive: https://github.com/VanLoo-lab/ascat/raw/master/ReleasedData/TCGA_SNP6_hg38/segments.tar.gz .

In [ ]:
directory = 'tcga_hg38'

tcga_hg38_samples = []

for filename in os.listdir(f'{data_path}/{directory}'):
    if filename.endswith(".txt"):
        df = pd.read_csv(os.path.join(directory, filename), sep='\t')      
        tcga_hg38_samples.append(df)
len(tcga_hg38_samples)

In [ ]:
tcga_hg38_dfs = pd.concat(tcga_hg38_samples)
tcga_hg38_dfs.head()

In [ ]:
tcga_hg38_dfs.columns = ["sample_id", "chrom", "start", "end", "major_cn", "minor_cn"]
tcga_hg38_dfs['chrom'] = 'chr' + tcga_hg38_dfs['chrom'].astype(str)
tcga_hg38_dfs.head()

In [ ]:
tcga_hg38_dfs.to_csv(f'{data_path}/tcga_hg38_cns_source.tsv', sep='\t', index=False)

# ENSAMBLE

In [14]:
# Install the assembly if missing
# pip install pyensembl
# !pyensembl install --release 75 --species homo_sapiens

from pyensembl import EnsemblRelease

In [17]:
# Specify the Ensembl release
data = EnsemblRelease(75)

genes = data.genes()

contig_names = list(map(str, range(1, 23))) + ['X', 'Y']

# Create a list of gene positions
gene_positions = [("chr" + gene.contig, gene.start, gene.end, gene.gene_id) for gene in genes if gene.biotype == 'protein_coding' and gene.contig in contig_names]

pos_df = pd.DataFrame(gene_positions, columns=["chrom", "start", "end", "name"])
pos_df["start"] = pos_df["start"] - 1
# sort by chrom, start

pos_df_sorted = pos_df.sort_values(by=['chrom', 'start'])
pos_df_sorted

,chrom,start,end,name
15704,chr1,69090,70008,ENSG00000186092
18842,chr1,134900,139379,ENSG00000237683
18769,chr1,367639,368634,ENSG00000235249
15471,chr1,621058,622053,ENSG00000185097
20205,chr1,738531,739137,ENSG00000269831
...,...,...,...,...
15948,chrY,26909215,26959626,ENSG00000187191
17865,chrY,26980007,27053183,ENSG00000205916
15650,chrY,27177047,27208695,ENSG00000185894
13145,chrY,27768263,27771049,ENSG00000172288


In [19]:
pos_df_sorted.to_csv(f'{data_path}/ENSEMBL_coding_genes.bed', index=False, sep="\t", header=False)